# Chapter 3 - Lab 2: Google ADK agent

**Goal**: build the agent with Google's **Agent Development Kit (ADK)**, running on Gemini.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **Google ADK** (`google-adk`) - `Agent`, `InMemoryRunner`, sessions.
* **Gemini** `gemini-2.5-flash` via `GOOGLE_API_KEY`.

## 1. Install packages

In [ ]:
%pip install -q google-adk google-genai pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('GOOGLE_API_KEY'), 'Set GOOGLE_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Define the tools and the agent

ADK reads plain Python functions as tools; type hints and docstrings become the schema.

In [ ]:
import os
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"   # API-key auth, not Vertex AI

from google.adk.agents import Agent

def get_support_data_tool(product_line: str) -> dict:
    """Get the quarter's support tickets and delivered orders for a product line (e.g. ZX1, FL-340)."""
    return get_support_data(product_line).model_dump()

def compute_contact_rate_tool(tickets: int, orders: int) -> float:
    """Compute the contact rate: support tickets per 100 orders."""
    return compute_contact_rate(tickets, orders)

agent = Agent(
    name="cx_analyst",
    model="gemini-2.5-flash",
    instruction=system_message,
    tools=[get_support_data_tool, compute_contact_rate_tool],
)

## 5. Run the agent with an in-memory session

In [ ]:
from google.adk.runners import InMemoryRunner
from google.genai import types

runner = InMemoryRunner(agent=agent)
session = await runner.session_service.create_session(app_name=runner.app_name, user_id="student")

content = types.Content(role="user", parts=[types.Part(text=input_message)])
async for event in runner.run_async(user_id="student", session_id=session.id, new_message=content):
    for part in (event.content.parts if event.content else []):
        if part.function_call:
            print(f"[Tool call] {part.function_call.name}({dict(part.function_call.args)})")
        if part.function_response:
            print(f"[Tool result] {part.function_response.response}")
    if event.is_final_response() and event.content:
        print("\n[Memo]\n" + "".join(p.text or "" for p in event.content.parts))

## 6. Results
**What to notice about ADK**: sessions and runners are first-class (enterprise deployment, Vertex AI Agent
Engine, evaluation tooling). Every step arrives as an *event*, which is useful for audit logs.